# 07 - Model Evaluation and Model Selection

This notebook evaluates the trained pipelines on the untouched test set.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc, precision_recall_curve, average_precision_score
)

project_root = Path("..").resolve()
split_artifact = joblib.load(project_root / "models" / "churn_train_test.joblib")
trained_pipelines = joblib.load(project_root / "models" / "churn_model_pipelines.joblib")
model_comparison = pd.read_csv(project_root / "models" / "model_comparison.csv", index_col="Model")

X_test = split_artifact["X_test"]
y_test = split_artifact["y_test"]
model_comparison

,Accuracy,Precision,Recall,F1-score,ROC-AUC
Model,,,,,
Logistic Regression,0.738112,0.504303,0.783422,0.613613,0.841639
Decision Tree,0.731725,0.494565,0.486631,0.490566,0.653696
Random Forest,0.783534,0.621053,0.473262,0.537178,0.822690


## Classification Reports and Confusion Matrices

`Yes=1` is the positive churn class. False negatives are churned customers the model misses; false positives are retained customers identified as churn risks.

In [2]:
evaluation_results = {}
for model_name, pipeline in trained_pipelines.items():
    predictions = pipeline.predict(X_test)
    probabilities = pipeline.predict_proba(X_test)[:, 1]
    evaluation_results[model_name] = {
        "predictions": predictions,
        "probabilities": probabilities
    }

    print("\n", model_name)
    print(classification_report(y_test, predictions, target_names=["Retained", "Churned"], zero_division=0))


 Logistic Regression
              precision    recall  f1-score   support

    Retained       0.90      0.72      0.80      1035
     Churned       0.50      0.78      0.61       374

    accuracy                           0.74      1409
   macro avg       0.70      0.75      0.71      1409
weighted avg       0.80      0.74      0.75      1409


 Decision Tree
              precision    recall  f1-score   support

    Retained       0.82      0.82      0.82      1035
     Churned       0.49      0.49      0.49       374

    accuracy                           0.73      1409
   macro avg       0.66      0.65      0.65      1409
weighted avg       0.73      0.73      0.73      1409




 Random Forest
              precision    recall  f1-score   support

    Retained       0.82      0.90      0.86      1035
     Churned       0.62      0.47      0.54       374

    accuracy                           0.78      1409
   macro avg       0.72      0.68      0.70      1409
weighted avg       0.77      0.78      0.77      1409



In [3]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (model_name, result) in zip(axes, evaluation_results.items()):
    cm = confusion_matrix(y_test, result["predictions"])
    ConfusionMatrixDisplay(cm, display_labels=["Retained", "Churned"]).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(model_name)
plt.tight_layout()
plt.savefig(project_root / "reports" / "confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

C:\Users\BHOOMIKA\AppData\Local\Temp\ipykernel_32164\1106796539.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## ROC Curves

ROC-AUC summarizes ranking performance across classification thresholds.

In [4]:
plt.figure(figsize=(8, 6))
for model_name, result in evaluation_results.items():
    fpr, tpr, _ = roc_curve(y_test, result["probabilities"])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{model_name} (AUC={roc_auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Random classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig(project_root / "reports" / "roc_curves.png", dpi=150, bbox_inches="tight")
plt.show()

C:\Users\BHOOMIKA\AppData\Local\Temp\ipykernel_32164\1129232519.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Precision-Recall Curves

Precision-recall curves are particularly informative for the imbalanced churn class.

In [5]:
plt.figure(figsize=(8, 6))
for model_name, result in evaluation_results.items():
    precision, recall, _ = precision_recall_curve(y_test, result["probabilities"])
    average_precision = average_precision_score(y_test, result["probabilities"])
    plt.plot(recall, precision, label=f"{model_name} (AP={average_precision:.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curves")
plt.legend()
plt.grid(alpha=0.3)
plt.savefig(project_root / "reports" / "precision_recall_curves.png", dpi=150, bbox_inches="tight")
plt.show()

C:\Users\BHOOMIKA\AppData\Local\Temp\ipykernel_32164\2423571223.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Model Comparison Visualization

In [6]:
model_comparison.plot(kind="bar", figsize=(12, 6))
plt.title("Machine Learning Model Comparison")
plt.ylabel("Score")
plt.xticks(rotation=45, ha="right")
plt.ylim(0, 1)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(project_root / "reports" / "model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

C:\Users\BHOOMIKA\AppData\Local\Temp\ipykernel_32164\1649740506.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## False Positive and False Negative Analysis

The confusion matrices below quantify the two main error types for each model.

In [7]:
error_rows = []
for model_name, result in evaluation_results.items():
    cm = confusion_matrix(y_test, result["predictions"])
    tn, fp, fn, tp = cm.ravel()
    error_rows.append({
        "Model": model_name,
        "True Negatives": tn,
        "False Positives": fp,
        "False Negatives": fn,
        "True Positives": tp
    })
pd.DataFrame(error_rows).set_index("Model")

,True Negatives,False Positives,False Negatives,True Positives
Model,,,,
Logistic Regression,747,288,81,293
Decision Tree,849,186,192,182
Random Forest,927,108,197,177


## Tree-Based Feature Importance

The transformed feature names include both scaled numerical names and one-hot categorical names.

In [8]:
preprocessor = trained_pipelines["Random Forest"].named_steps["preprocessor"]
feature_names = preprocessor.get_feature_names_out()

tree_models = ["Decision Tree", "Random Forest"]
for model_name in tree_models:
    classifier = trained_pipelines[model_name].named_steps["classifier"]
    importance = pd.Series(classifier.feature_importances_, index=feature_names).sort_values(ascending=False)
    plt.figure(figsize=(10, 7))
    importance.head(15).sort_values().plot(kind="barh")
    plt.title(f"Top 15 Feature Importances - {model_name}")
    plt.xlabel("Importance")
    plt.tight_layout()
    plt.savefig(project_root / "reports" / f"{model_name.lower().replace(' ', '_')}_feature_importance.png", dpi=150, bbox_inches="tight")
    plt.show()

C:\Users\BHOOMIKA\AppData\Local\Temp\ipykernel_32164\1903417512.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


C:\Users\BHOOMIKA\AppData\Local\Temp\ipykernel_32164\1903417512.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Logistic Regression Coefficient Interpretation

A positive coefficient increases predicted log-odds of churn; a negative coefficient decreases it.

In [9]:
logistic_pipeline = trained_pipelines["Logistic Regression"]
logistic_preprocessor = logistic_pipeline.named_steps["preprocessor"]
logistic_classifier = logistic_pipeline.named_steps["classifier"]
logistic_feature_names = logistic_preprocessor.get_feature_names_out()
coefficient_table = pd.Series(logistic_classifier.coef_[0], index=logistic_feature_names).sort_values()

top_logistic = pd.concat([coefficient_table.head(10), coefficient_table.tail(10)])
plt.figure(figsize=(10, 8))
top_logistic.plot(kind="barh")
plt.axvline(0, color="black", linewidth=1)
plt.title("Logistic Regression Coefficients")
plt.xlabel("Coefficient")
plt.tight_layout()
plt.savefig(project_root / "reports" / "logistic_coefficients.png", dpi=150, bbox_inches="tight")
plt.show()

coefficient_table.to_csv(project_root / "models" / "logistic_coefficients.csv")

C:\Users\BHOOMIKA\AppData\Local\Temp\ipykernel_32164\2742635643.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Best Model Selection

The best model is selected primarily using the ranking metrics from the test set. A final choice should also consider recall for identifying actual churners, since missed churners can represent lost future revenue.

In [10]:
selection_score = model_comparison.copy()
selection_score["Average of Recall and ROC-AUC"] = (
    selection_score["Recall"] + selection_score["ROC-AUC"]
) / 2
selection_score.sort_values("Average of Recall and ROC-AUC", ascending=False)

,Accuracy,Precision,Recall,F1-score,ROC-AUC,Average of Recall and ROC-AUC
Model,,,,,,
Logistic Regression,0.738112,0.504303,0.783422,0.613613,0.841639,0.812531
Random Forest,0.783534,0.621053,0.473262,0.537178,0.822690,0.647976
Decision Tree,0.731725,0.494565,0.486631,0.490566,0.653696,0.570163


In [11]:
best_model_name = selection_score["Average of Recall and ROC-AUC"].idxmax()
print("Selected model:", best_model_name)
print(selection_score.loc[best_model_name].to_string())

selected_complete_pipeline = trained_pipelines[best_model_name]
joblib.dump(selected_complete_pipeline, project_root / "models" / "churn_model.joblib")

Selected model: Logistic Regression
Accuracy                         0.738112
Precision                        0.504303
Recall                           0.783422
F1-score                         0.613613
ROC-AUC                          0.841639
Average of Recall and ROC-AUC    0.812531


['C:\\Users\\BHOOMIKA\\Desktop\\Customer-Churn-Analysis\\models\\churn_model.joblib']

## Reload and Predict Raw Customer Records

The saved pipeline includes preprocessing and the classifier, so it accepts raw customer feature rows and performs one-hot encoding and scaling automatically.

In [12]:
raw_df = pd.read_csv(project_root / "dataset" / "processed" / "cleaned_telco.csv")
raw_features = raw_df.drop(columns=["customerID", "Churn"])
sample_customers = raw_features.head(5)

reloaded_pipeline = joblib.load(project_root / "models" / "churn_model.joblib")
sample_predictions = reloaded_pipeline.predict(sample_customers)
sample_probabilities = reloaded_pipeline.predict_proba(sample_customers)[:, 1]

sample_results = sample_customers.copy()
sample_results["PredictedChurn"] = np.where(sample_predictions == 1, "Yes", "No")
sample_results["ChurnProbability"] = sample_probabilities
sample_results.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,PredictedChurn,ChurnProbability
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,...,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,Yes,0.806413
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,...,No,No,No,One year,No,Mailed check,56.95,1889.50,No,0.117771
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,0.531999
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,...,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,0.080972
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,0.854480
